# Tool: Búsqueda en web

Aquí la búsqueda no es una función tuya: es un plugin que OpenRouter corre por ti. Haces la misma pregunta con y sin él, y revisas las fuentes.

In [ ]:
import os
import json

import httpx
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.environ.get("OPENROUTER_API_KEY")
assert API_KEY, "Falta OPENROUTER_API_KEY"

## La declaración

En el taller de la fecha tú escribías la función y tú la ejecutabas. La búsqueda web es distinta: se declara en el body con la clave `plugins` y el servidor de OpenRouter hace la búsqueda antes de que el modelo responda.

`max_results` dice cuántas páginas trae. Si prefieres algo aún más corto, existe el atajo de ponerle el sufijo `:online` al nombre del modelo.

In [ ]:
plugins = [
    {
        "id": "web",
        "max_results": 3,
    }
]

print(json.dumps(plugins, indent=2, ensure_ascii=False))

## Con búsqueda

El mismo `POST` de siempre a `chat/completions`, con una llave nueva en el body: `plugins`.

In [ ]:
respuesta_con_web = httpx.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={
        "model": "nvidia/nemotron-3-super-120b-a12b:free",
        "messages": [
            {
                "role": "user",
                "content": "¿Qué noticias de tecnología salieron esta semana? Dame 3 titulares con su fecha.",
            }
        ],
        "plugins": plugins,
    },
    timeout=120,
)
respuesta_con_web.raise_for_status()

print(respuesta_con_web.json()["choices"][0]["message"]["content"])

## Sin búsqueda

La misma pregunta, el mismo modelo, pero sin la llave `plugins`. Ahora el modelo solo tiene lo que memorizó cuando lo entrenaron.

In [ ]:
respuesta_sin_web = httpx.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={
        "model": "nvidia/nemotron-3-super-120b-a12b:free",
        "messages": [
            {
                "role": "user",
                "content": "¿Qué noticias de tecnología salieron esta semana? Dame 3 titulares con su fecha.",
            }
        ],
    },
    timeout=120,
)
respuesta_sin_web.raise_for_status()

print(respuesta_sin_web.json()["choices"][0]["message"]["content"])

## Las fuentes

Cuando el plugin trabaja, el mensaje del modelo trae una lista extra llamada `annotations`. Cada entrada es una cita con el título de la página, su URL y el pedazo de texto que se usó.

La respuesta sin búsqueda no trae nada de eso: por eso usamos `.get("annotations", [])`.

In [ ]:
anotaciones = respuesta_con_web.json()["choices"][0]["message"].get("annotations", [])

print(len(anotaciones), "fuentes con búsqueda")
print(len(respuesta_sin_web.json()["choices"][0]["message"].get("annotations", [])), "fuentes sin búsqueda")

for anotacion in anotaciones:
    print()
    print(anotacion["url_citation"]["title"])
    print(anotacion["url_citation"]["url"])

In [ ]:
print(json.dumps(anotaciones[0], indent=2, ensure_ascii=False))